In [1]:
print("Hello")

Hello


In [2]:
!python -m pip install pyshark

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [pyshark]
   ---------------------------------------- 2/2 [pyshark]




[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pyshark

interfaces = pyshark.tshark.tshark.get_tshark_interfaces()

print("Available Interfaces:")
for iface in interfaces:
    print(iface)

Available Interfaces:
\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}
\Device\NPF_{86DB4D00-64E0-47F7-A8A3-0E4B15A706AF}
\Device\NPF_{9EFB40FF-BBC7-4B57-942A-6AC979719858}
\Device\NPF_{A3DB423A-5E9C-48B7-8DED-E03E46F06A5B}
\Device\NPF_{E469F6EB-35DF-4D30-815E-7CB09E606936}
\Device\NPF_{3F298E2B-FE46-4581-BEFA-A08694E0FBE4}
\Device\NPF_{9ACC5697-0F79-4C2A-B9D7-F89CC314D5EC}
\Device\NPF_{EE5D58FE-35D0-4B0C-8825-A527F098AC4B}
\Device\NPF_Loopback
etwdump


In [4]:
import pyshark

capture = pyshark.LiveCapture(interface="Wi-Fi")

print("Capturing 10 packets...\n")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)

C:\Users\v-yaalam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyshark\capture\capture.py:177: UserWarning: The running eventloop has tasks but pyshark must set a new eventloop to continue. Existing tasks may not run.
  warnings.warn("The running eventloop has tasks but pyshark must set a new eventloop to continue. "


Capturing 10 packets...



RuntimeError: Cannot run the event loop while another loop is running

In [5]:
import pyshark

capture = pyshark.LiveCapture(interface="Wi-Fi")

print("Capturing packets...\n")

for packet in capture.sniff_continuously(packet_count=20):

    try:
        src_ip = packet.ip.src
        dst_ip = packet.ip.dst
        protocol = packet.highest_layer

        print(f"{src_ip} -> {dst_ip} | Protocol: {protocol}")

    except AttributeError:
        print("Non-IP packet captured")

Capturing packets...



RuntimeError: Cannot run the event loop while another loop is running

Yes, it is possible to capture packets from your own device using Python in VS Code. For learning, the safest beginner path is **PyShark + Wireshark/TShark**, and the more Python-native path is **Scapy**.

I also checked available references. Official-style guidance says **PyShark LiveCapture captures from a live network interface and supports packet count, timeout, BPF filters, display filters, and output files**. Usage — Scapy documentation shows Scapy can be used from Python and notes that packet operations on Windows usually need administrator privileges. I also found internal learning/resources around Python packet analysis and Scapy, including [Network Activity and Packet Analysis with Python](https://learning.cloud.microsoft/detail/7958e409-5c14-459a-8e76-eb5059ae4e20?context={%22subEntityId%22:{%22source%22:%22M365Search%22}}\&EntityRepresentationId=46ff4307-37af-4ae6-a325-2d4dd22d7cc2), and internal packet-capture troubleshooting references. The Microsoft 365 search returned broad results across file/external domains, including packet capture, Scapy, PyShark, tcpdump, and Wireshark resources. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/capture_usage/live_capture_usage/), [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/parameters/live_capture_parameters/) [\[scapy.readthedocs.io\]](https://scapy.readthedocs.io/en/latest/usage.html) [\[Network Ac...ith Python \| Viva Learning\]](https://learning.cloud.microsoft/detail/7958e409-5c14-459a-8e76-eb5059ae4e20?context={%22subEntityId%22:{%22source%22:%22M365Search%22}}), [\[1 - Network Traces \| PowerPoint\]](https://microsoft.sharepoint.com/teams/Azure_IaaS_Technical_Support/_layouts/15/Doc.aspx?sourcedoc=%7BCB91C8FE-7CF3-49C5-92EE-E4CE9F926E5A%7D&file=1%20-%20Network%20Traces.pptx&action=edit&mobileredirect=true&DefaultItemOpen=1)

> Important: run captures only on your own device or networks where you have permission. Packet captures can include sensitive data like DNS queries, IPs, ports, and sometimes payload metadata.

***

# Option 1: Best Beginner Method: PyShark

PyShark is easy because it uses Wireshark/TShark underneath. PyShark documentation says it is a wrapper for TShark and can parse live captures using Wireshark dissectors. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/capture_usage/live_capture_usage/), [\[kiminewt.github.io\]](https://kiminewt.github.io/pyshark/)

## Step 1: Install Wireshark

Install Wireshark and make sure **Npcap** is selected during installation.

After installing, open PowerShell and verify:

```powershell
tshark -v
```

If that command works, PyShark can use TShark.

***

## Step 2: Install PyShark

In VS Code terminal:

```bash
python -m pip install pyshark
```

***

## Step 3: Find Your Network Interfaces

Create a file:

```python
import pyshark

interfaces = pyshark.tshark.tshark.get_tshark_interfaces()

print("Available Interfaces:")
for iface in interfaces:
    print(iface)
```

Run:

```bash
python interfaces.py
```

You may see output like:

```text
\Device\NPF_{XXXXXX}
Wi-Fi
Ethernet
Loopback
```

On Windows, interface names can look long. If `Wi-Fi` works, use it. If not, use the full interface name from the output.

***

# Example 1: Capture 10 Packets and Print Summary

```python
import pyshark

capture = pyshark.LiveCapture(interface="Wi-Fi")

print("Capturing 10 packets...\n")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)
```

### What you learn here

* `LiveCapture()` starts packet capture
* `interface="Wi-Fi"` tells Python which NIC to capture from
* `packet_count=10` stops after 10 packets
* `print(packet)` gives packet details

PyShark supports `sniff_continuously(packet_count=10)` for limited packet capture. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/capture_usage/live_capture_usage/)

***

# Example 2: Capture and Print Source IP, Destination IP, Protocol

```python
import pyshark

capture = pyshark.LiveCapture(interface="Wi-Fi")

print("Capturing packets...\n")

for packet in capture.sniff_continuously(packet_count=20):

    try:
        src_ip = packet.ip.src
        dst_ip = packet.ip.dst
        protocol = packet.highest_layer

        print(f"{src_ip} -> {dst_ip} | Protocol: {protocol}")

    except AttributeError:
        print("Non-IP packet captured")
```

### Example Output

```text
192.168.1.10 -> 8.8.8.8 | Protocol: DNS
192.168.1.10 -> 13.107.246.40 | Protocol: TLS
Non-IP packet captured
```

### What you learn

* Some packets are IP packets
* Some packets are ARP, STP, LLDP, etc.
* `try/except` prevents the script from crashing

***

# Example 3: Capture Only DNS Packets

DNS usually uses UDP port 53.

```python
import pyshark

capture = pyshark.LiveCapture(
    interface="Wi-Fi",
    bpf_filter="udp port 53"
)

print("Capturing DNS packets...\n")

for packet in capture.sniff_continuously(packet_count=10):

    try:
        print("Source IP     :", packet.ip.src)
        print("Destination IP:", packet.ip.dst)
        print("Protocol      :", packet.highest_layer)
        print("-" * 40)

    except AttributeError:
        pass
```

PyShark supports `bpf_filter` to prefilter captured packets. The documentation gives DNS capture examples using `udp port 53`. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/capture_usage/live_capture_usage/), [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/parameters/live_capture_parameters/)

***

# Example 4: Capture Packets and Save to PCAP File

This is useful because you can open the file in Wireshark.

```python
import pyshark

capture = pyshark.LiveCapture(
    interface="Wi-Fi",
    output_file="my_capture.pcap"
)

print("Capturing 20 packets and saving to my_capture.pcap")

for packet in capture.sniff_continuously(packet_count=20):
    print(packet.highest_layer)

print("Capture completed")
```

After running the script, open this file in Wireshark:

```text
my_capture.pcap
```

PyShark LiveCapture supports an `output_file` parameter to save captured packets. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/parameters/live_capture_parameters/)

***

# Example 5: Capture TCP Port 443 Traffic

This captures HTTPS traffic metadata.

```python
import pyshark

capture = pyshark.LiveCapture(
    interface="Wi-Fi",
    bpf_filter="tcp port 443"
)

print("Capturing HTTPS/TLS packets...\n")

for packet in capture.sniff_continuously(packet_count=15):

    try:
        print(
            packet.ip.src,
            "->",
            packet.ip.dst,
            "|",
            packet.transport_layer,
            "|",
            packet.highest_layer
        )

    except AttributeError:
        pass
```

### Example Output

```text
192.168.1.10 -> 20.189.173.15 | TCP | TLS
20.189.173.15 -> 192.168.1.10 | TCP | TLS
```

### Important

You usually cannot see encrypted HTTPS content. You mostly see:

* Source IP
* Destination IP
* Port
* TCP flags
* TLS metadata

***

# Option 2: Scapy Packet Capture

Scapy is more direct and Python-native. Scapy documentation describes Scapy as a packet manipulation tool that can build, send, receive, dissect, and capture packets. [\[scapy.readthedocs.io\]](https://scapy.readthedocs.io/en/latest/index.html), [\[github.com\]](https://github.com/secdev/scapy)

## Install Scapy

```bash
python -m pip install scapy
```

On Windows, also install Npcap through Wireshark installation.

***

# Example 6: Basic Scapy Sniffer

```python
from scapy.all import sniff

def show_packet(packet):
    print(packet.summary())

sniff(
    count=10,
    prn=show_packet
)
```

### What it does

* Captures 10 packets
* Calls `show_packet()` for every packet
* Prints a short packet summary

Scapy’s `sniff()` supports `count`, `filter`, `iface`, `prn`, and timeout-style usage. [\[0xbharath.github.io\]](https://0xbharath.github.io/art-of-packet-crafting-with-scapy/scapy/sniffing/index.html), [\[geeksforgeeks.org\]](https://www.geeksforgeeks.org/python/packet-sniffing-using-scapy/)

***

# Example 7: Scapy Capture Only ICMP/Ping Packets

Open one terminal and run this Python script:

```python
from scapy.all import sniff

def show_packet(packet):
    print(packet.summary())

sniff(
    filter="icmp",
    count=5,
    prn=show_packet
)
```

Open another terminal and run:

```bash
ping 8.8.8.8
```

You should see ICMP packets in Python.

***

# Example 8: Scapy Capture TCP Packets

```python
from scapy.all import sniff, IP, TCP

def show_tcp(packet):
    if IP in packet and TCP in packet:
        print(
            packet[IP].src,
            "->",
            packet[IP].dst,
            "TCP",
            packet[TCP].sport,
            "->",
            packet[TCP].dport
        )

sniff(
    filter="tcp",
    count=20,
    prn=show_tcp
)
```

### Example Output

```text
192.168.1.10 -> 13.107.246.40 TCP 51544 -> 443
13.107.246.40 -> 192.168.1.10 TCP 443 -> 51544
```

***

# Example 9: Save Scapy Capture to PCAP

```python
from scapy.all import sniff, wrpcap

print("Capturing 20 packets...")

packets = sniff(count=20)

wrpcap("scapy_capture.pcap", packets)

print("Saved packets to scapy_capture.pcap")
```

Scapy can save captured packets to a PCAP file using `wrpcap()`. [\[geeksforgeeks.org\]](https://www.geeksforgeeks.org/python/packet-sniffing-using-scapy/)

***

# Example 10: Read PCAP File Using Scapy

After saving a `.pcap`, read it back:

```python
from scapy.all import rdpcap

packets = rdpcap("scapy_capture.pcap")

print("Total packets:", len(packets))

for packet in packets:
    print(packet.summary())
```

### What you learn

* How to capture packets
* How to save packets
* How to read packet captures later
* How tools like Wireshark store capture data

***

# Example 11: Count Protocols in a PCAP File

```python
from scapy.all import rdpcap, IP, TCP, UDP, ICMP

packets = rdpcap("scapy_capture.pcap")

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

for packet in packets:

    if TCP in packet:
        tcp_count += 1

    elif UDP in packet:
        udp_count += 1

    elif ICMP in packet:
        icmp_count += 1

    else:
        other_count += 1

print("TCP packets :", tcp_count)
print("UDP packets :", udp_count)
print("ICMP packets:", icmp_count)
print("Other       :", other_count)
```

### Example Output

```text
TCP packets : 15
UDP packets : 4
ICMP packets: 1
Other       : 0
```

This is a very good beginner automation example because you are using:

* Variables
* Loops
* Conditions
* Packet parsing
* Counters

***

# Example 12: Create a Simple Packet Capture Analyzer

This is closer to network engineer style.

```python
from scapy.all import sniff, IP, TCP, UDP, ICMP

def analyze_packet(packet):

    if IP in packet:

        src = packet[IP].src
        dst = packet[IP].dst

        if TCP in packet:
            print(f"TCP  {src} -> {dst}")

        elif UDP in packet:
            print(f"UDP  {src} -> {dst}")

        elif ICMP in packet:
            print(f"ICMP {src} -> {dst}")

        else:
            print(f"IP   {src} -> {dst}")

    else:
        print("Non-IP packet:", packet.summary())


sniff(
    count=30,
    prn=analyze_packet
)
```

***

# Example 13: Capture Packets for a Specific Host

Example: capture traffic to/from `8.8.8.8`.

```python
from scapy.all import sniff

def show_packet(packet):
    print(packet.summary())

sniff(
    filter="host 8.8.8.8",
    count=20,
    prn=show_packet
)
```

You can test with:

```bash
ping 8.8.8.8
```

***

# Example 14: Capture DNS Queries and Print Domain Names with PyShark

```python
import pyshark

capture = pyshark.LiveCapture(
    interface="Wi-Fi",
    display_filter="dns"
)

print("Capturing DNS queries...\n")

for packet in capture.sniff_continuously(packet_count=10):

    try:
        print("Source IP:", packet.ip.src)
        print("DNS Query:", packet.dns.qry_name)
        print("-" * 40)

    except AttributeError:
        pass
```

PyShark supports `display_filter`, and the documentation shows DNS examples using display filters. [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/capture_usage/live_capture_usage/), [\[pyshark-pa...thedocs.io\]](https://pyshark-packet-analysis.readthedocs.io/en/latest/parameters/live_capture_parameters/)

***

# Example 15: Use Python to Run TShark Directly

This is useful if PyShark has issues.

```python
import subprocess

command = [
    "tshark",
    "-i", "Wi-Fi",
    "-c", "10"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print(result.stdout)
```

### DNS-only capture:

```python
import subprocess

command = [
    "tshark",
    "-i", "Wi-Fi",
    "-f", "udp port 53",
    "-c", "10"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print(result.stdout)
```

Internal training material also refers to TShark as Wireshark’s command-line executable and shows it can capture traffic and write capture files. [\[1 - Network Traces \| PowerPoint\]](https://microsoft.sharepoint.com/teams/Azure_IaaS_Technical_Support/_layouts/15/Doc.aspx?sourcedoc=%7BCB91C8FE-7CF3-49C5-92EE-E4CE9F926E5A%7D&file=1%20-%20Network%20Traces.pptx&action=edit&mobileredirect=true&DefaultItemOpen=1)

***

# Which One Should You Use?

## Use PyShark if you want:

* Easier packet reading
* Wireshark-style decoding
* DNS/TLS/HTTP protocol parsing
* Better beginner experience

## Use Scapy if you want:

* More Python control
* Packet creation
* Packet sending
* Custom parsing
* Automation style learning

## Use TShark directly if:

* PyShark has errors
* You just want quick captures
* You want command-line style output

***

# VS Code Setup Checklist

Run VS Code as Administrator.

Install packages:

```bash
python -m pip install pyshark scapy
```

Verify TShark:

```bash
tshark -v
```

Create file:

```text
packet_capture_test.py
```

Run:

```bash
python packet_capture_test.py
```

If capture does not work:

```bash
python -m pip show pyshark
python -m pip show scapy
tshark -D
```

`tshark -D` lists interfaces.

***

# Common Errors and Fixes

## Error 1: `tshark not found`

Fix:

* Install Wireshark
* Add Wireshark folder to PATH
* Usually path is:

```text
C:\Program Files\Wireshark
```

Then restart VS Code.

***

## Error 2: No packets captured

Fix:

* Run VS Code as Administrator
* Check correct interface name
* Try `tshark -D`
* Replace `"Wi-Fi"` with actual interface name

***

## Error 3: Permission denied

Fix:

* Run terminal as Administrator
* On Linux/macOS, use sudo:

```bash
sudo python capture.py
```

Scapy documentation notes root/admin-style privileges are needed for some packet operations. [\[scapy.readthedocs.io\]](https://scapy.readthedocs.io/en/latest/usage.html)

***

## Error 4: Interface name not working

Run:

```bash
tshark -D
```

Example output:

```text
1. \Device\NPF_{ABC}
2. Wi-Fi
3. Ethernet
```

Then use:

```python
capture = pyshark.LiveCapture(interface="2")
```

or:

```python
capture = pyshark.LiveCapture(interface="Wi-Fi")
```

***

# My Recommended Learning Order for You

Since you are learning Python with networking, follow this order:

1. Capture 10 packets and print summaries
2. Capture only ICMP and test with ping
3. Capture only DNS
4. Save capture to `.pcap`
5. Open `.pcap` in Wireshark
6. Read `.pcap` back using Python
7. Count TCP/UDP/ICMP packets
8. Build a mini network troubleshooting report

***

# Mini Project: Packet Capture Health Report

Try this full beginner project:

```python
from scapy.all import sniff, IP, TCP, UDP, ICMP, wrpcap

packet_list = []

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

def analyze(packet):
    global tcp_count, udp_count, icmp_count, other_count

    packet_list.append(packet)

    if IP in packet:

        src = packet[IP].src
        dst = packet[IP].dst

        if TCP in packet:
            tcp_count += 1
            print(f"TCP  {src} -> {dst}")

        elif UDP in packet:
            udp_count += 1
            print(f"UDP  {src} -> {dst}")

        elif ICMP in packet:
            icmp_count += 1
            print(f"ICMP {src} -> {dst}")

        else:
            other_count += 1
            print(f"IP   {src} -> {dst}")

    else:
        other_count += 1
        print("Non-IP:", packet.summary())


print("=" * 50)
print("PACKET CAPTURE STARTED")
print("=" * 50)

sniff(
    count=30,
    prn=analyze
)

wrpcap("network_report_capture.pcap", packet_list)

print("\n" + "=" * 50)
print("PACKET CAPTURE SUMMARY")
print("=" * 50)

print("TCP packets :", tcp_count)
print("UDP packets :", udp_count)
print("ICMP packets:", icmp_count)
print("Other       :", other_count)

print("\nSaved file: network_report_capture.pcap")
```

This is a very good first practical packet capture automation script. It captures packets, analyzes basic protocols, prints output, and saves a PCAP file that you can open in Wireshark.


In [6]:
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

print("Capturing 10 packets...\n")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)

Capturing 10 packets...



RuntimeError: Cannot run the event loop while another loop is running

I can see the exact problem from your screenshots.

### Good News ✅

* `pyshark` installed successfully.
* `tshark` is working.
* Interfaces are being detected.
* The error is **NOT** a missing package issue anymore.

### Actual Error

```text
RuntimeError:
Cannot run the event loop while another loop is running
```

This happens because you're running PyShark inside a **Jupyter Notebook cell** (VS Code Notebook).

PyShark tries to create its own asyncio event loop, but Jupyter already has one running.

***

# Fix 1 (Recommended): Run as a .py File

Create:

```text
capture.py
```

Put this code in it:

```python
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

print("Capturing 10 packets...\n")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)
```

Run from terminal:

```bash
python capture.py
```

not

```python
# Jupyter Cell
```

PyShark works much better this way.

***

# Fix 2: Use nest\_asyncio in Notebook

Install:

```bash
pip install nest-asyncio
```

Then first notebook cell:

```python
import nest_asyncio
nest_asyncio.apply()
```

Now try:

```python
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)
```

Many Jupyter users need this workaround.

***

# Fix 3: Don't Use "Wi-Fi"

Your available interfaces are:

```text
\Device\NPF_{0E6D5B36-...}
\Device\NPF_{86DB4D00-...}
...
\Device\NPF_Loopback
```

But your code uses:

```python
interface="Wi-Fi"
```

Use one of the actual interfaces found:

```python
capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)
```

***

# Find Which Interface is Wi-Fi

Open Administrator PowerShell:

```powershell
tshark -D
```

Example:

```text
1. Ethernet
2. Wi-Fi
3. Loopback
```

Then:

```python
capture = pyshark.LiveCapture(
    interface="2"
)
```

or

```python
capture = pyshark.LiveCapture(
    interface="Wi-Fi"
)
```

if tshark shows that exact name.

***

# Simpler Test Before Capturing

Run:

```python
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

capture.sniff(timeout=5)

print("Packets Captured:", len(capture))
```

If this works:

```text
Packets Captured: 25
```

then PyShark is functioning.

***

# Alternative: Use Scapy (Easier for Learning)

Install:

```bash
pip install scapy
```

Then:

```python
from scapy.all import sniff

def packet_callback(packet):
    print(packet.summary())

sniff(
    count=10,
    prn=packet_callback
)
```

Works very well in VS Code and is commonly used by network engineers for automation.

***

# One More Potential Problem

I noticed you're using:

```text
Python 3.13
```

PyShark generally works, but some networking libraries still have quirks on 3.13.

If you continue getting strange errors:

```bash
py -3.12 -m venv packetlab

packetlab\Scripts\activate

pip install pyshark
pip install scapy
```

Python 3.12 is currently safer for packet-capture labs.

### Try this first in the notebook:

```python
import nest_asyncio
nest_asyncio.apply()
```

and replace `"Wi-Fi"` with the full `\Device\NPF_{...}` interface from your list. That should resolve the event-loop error you're seeing.


In [7]:
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

print("Capturing packets...")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)

Capturing packets...


RuntimeError: Cannot run the event loop while another loop is running

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\re\__init__.py:310: RuntimeWarning: coroutine 'LiveCapture._get_tshark_process' was never awaited
  return pattern.translate(_special_chars_map)


In [8]:
!pip install nest-asyncio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import nest_asyncio

nest_asyncio.apply()

In [10]:
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)

RuntimeError: Cannot run the event loop while another loop is running

In [11]:
!pip install scapy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
from scapy.all import sniff

sniff(
    count=10,
    prn=lambda pkt: print(pkt.summary())
)

C:\Users\v-yaalam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scapy\base_classes.py:415: RuntimeWarning: coroutine 'LiveCapture._get_tshark_process' was never awaited
  dct["_%s" % attr] = dct.pop(attr)


Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw


<Sniffed: TCP:0 UDP:10 ICMP:0 Other:0>

In [13]:
import nest_asyncio
nest_asyncio.apply()

import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

capture.sniff(timeout=5)

print("Packets Captured:", len(capture))

RuntimeError: Cannot run the event loop while another loop is running

You're still getting the **same Jupyter Notebook problem**, not a PyShark problem.

### Root Cause

The error says:

```text
RuntimeError:
Cannot run the event loop while another loop is running
```

This happens because:

```text
VS Code Notebook (Jupyter)
        ↓
Already running asyncio loop
        ↓
PyShark tries to create another loop
        ↓
Boom 💥
```

Your interface is correct now:

```python
interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
```

The issue is **Jupyter Notebook + PyShark**.

***

# Fix Option 1 (Best)

Create a normal Python file:

```text
capture.py
```

Put:

```python
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

print("Capturing packets...")

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)
```

Run from terminal:

```bash
python capture.py
```

NOT from a notebook cell.

***

# Fix Option 2 (Notebook Workaround)

Install:

```bash
pip install nest-asyncio
```

First notebook cell:

```python
import nest_asyncio

nest_asyncio.apply()
```

Second cell:

```python
import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

for packet in capture.sniff_continuously(packet_count=10):
    print(packet)
```

***

# Even Better for Learning: Use Scapy

For packet capture learning, I recommend Scapy over PyShark.

Install:

```bash
pip install scapy
```

Test:

```python
from scapy.all import sniff

sniff(
    count=10,
    prn=lambda pkt: print(pkt.summary())
)
```

This works much better in VS Code and teaches networking fundamentals directly.

***

# Verify Tshark First

Run in terminal:

```bash
tshark -D
```

Run:

```bash
tshark -i 1 -c 5
```

or

```bash
tshark -i 2 -c 5
```

until you see packets.

Example:

```text
1. Ethernet
2. Wi-Fi
3. Loopback
```

Then use that same interface in PyShark.

***

# Quick Notebook Fix Code

Copy this exactly into a notebook cell:

```python
import nest_asyncio
nest_asyncio.apply()

import pyshark

capture = pyshark.LiveCapture(
    interface=r"\Device\NPF_{0E6D5B36-2E21-4586-A7D1-BB1D1F54EDA2}"
)

capture.sniff(timeout=5)

print("Packets Captured:", len(capture))
```

If that still fails, switch to a normal `.py` file, which is the most reliable fix for this specific error.


In [14]:
from scapy.all import sniff

def show_packet(packet):
    print(packet.summary())

sniff(
    count=10,
    prn=show_packet
)

Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw / Padding
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw


<Sniffed: TCP:0 UDP:10 ICMP:0 Other:0>

In [ ]:
from scapy.all import sniff

sniff(
    filter="icmp",
    count=10,
    prn=lambda pkt: print(pkt.summary())
)

In [12]:
from scapy.all import sniff, IP

def analyze(packet):

    if IP in packet:

        print(
            "Source:",
            packet[IP].src,
            "->",
            "Destination:",
            packet[IP].dst
        )

sniff(
    count=20,
    prn=analyze
)

Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 51.5.71.12 -> Destination: 10.0.9.200
Source: 51.5.71.12 -> Destination: 10.0.9.200
Source: 51.5.71.12 -> Destination: 10.0.9.200
Source: 51.5.71.36 -> Destination: 10.0.9.200
Source: 51.5.71.36 -> Destination: 10.0.9.200
Source: 51.5.71.36 -> Destination: 10.0.9.200
Source: 51.5.71.27 -> Destination: 10.0.9.200
Source: 51.5.71.36 -> Destination: 10.0.9.200
Source: 40.64.145.160 -> Destination: 10.0.9.200
Source: 40.64.145.160 -> Destination: 10.0.9.200
Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 51.5.71.12 -> Destination: 10.0.9.200
Source: 10.0.9.200 -> Destination: 51.5.71.36
Source: 51.5.71.12 -> Destination: 10.0.9.200
Source: 10.0.9.200 -> Destination: 40.64.145.160
Source: 10.0.9.200 -> Destination: 40.64.145.160


<Sniffed: TCP:4 UDP:16 ICMP:0 Other:0>

In [11]:
from scapy.all import sniff, IP, TCP

def analyze(packet):

    if TCP in packet:

        print(
            packet[IP].src,
            ":",
            packet[TCP].sport,
            "-->",
            packet[IP].dst,
            ":",
            packet[TCP].dport
        )

sniff(
    filter="tcp",
    count=20,
    prn=analyze
)

40.64.145.160 : 443 --> 10.0.9.200 : 62581
52.107.248.17 : 443 --> 10.0.9.200 : 64992
10.0.9.200 : 64992 --> 52.107.248.17 : 443
104.18.39.21 : 443 --> 10.0.9.200 : 52706
10.0.9.200 : 52706 --> 104.18.39.21 : 443
20.59.87.225 : 443 --> 10.0.9.200 : 59450
10.0.9.200 : 59450 --> 20.59.87.225 : 443
10.0.1.40 : 58446 --> 10.0.9.200 : 7680
10.0.9.200 : 7680 --> 10.0.1.40 : 58446
10.0.1.40 : 58446 --> 10.0.9.200 : 7680
10.0.1.40 : 58446 --> 10.0.9.200 : 7680
10.0.9.200 : 7680 --> 10.0.1.40 : 58446
10.0.1.40 : 58446 --> 10.0.9.200 : 7680
10.0.1.40 : 58446 --> 10.0.9.200 : 7680
10.0.9.200 : 7680 --> 10.0.1.40 : 58446
52.123.187.6 : 443 --> 10.0.9.200 : 56266
10.0.9.200 : 56266 --> 52.123.187.6 : 443
10.0.9.200 : 56783 --> 104.208.16.94 : 443
104.208.16.94 : 443 --> 10.0.9.200 : 56783
40.64.145.160 : 443 --> 10.0.9.200 : 62583


<Sniffed: TCP:20 UDP:0 ICMP:0 Other:0>

In [10]:
from scapy.all import sniff

sniff(
    filter="udp port 53",
    count=20,
    prn=lambda pkt: print(pkt.summary())
)

Ether / IP / UDP / DNS Qry b'alertus.corp.microsoft.com.'
Ether / IP / UDP / DNS Ans name-error
Ether / IP / UDP / DNS Qry b'b1.nel.goog.'
Ether / IP / UDP / DNS Ans 142.250.100.94
Ether / IP / UDP / DNS Qry b'unitedstates.smartscreen.microsoft.com.'
Ether / IP / UDP / DNS Ans b'prod-atm-wds-e5-unitedstates2.trafficmanager.net.'
Ether / IP / UDP / DNS Qry b'23fdd92b-dfef-4922-bbac-2c4d529e8b3f.ods.opinsights.azure.com.'
Ether / IP / UDP / DNS Ans b'usw3-oi-ods.trafficmanager.net.'
Ether / IP / UDP / DNS Qry b'displaycatalog.mp.microsoft.com.'
Ether / IP / UDP / DNS Ans b'consumer-prod-bcfd-v7-afd.trafficmanager.net.'
Ether / IP / UDP / DNS Qry b'collections.mp.microsoft.com.'
Ether / IP / UDP / DNS Ans b'consumer-collections-aks2eap.md.mp.microsoft.com.akadns.net.'
Ether / IP / UDP / DNS Qry b'spclient.wg.spotify.com.'
Ether / IP / UDP / DNS Ans b'edge-web.dual-gslb.spotify.com.'
Ether / IP / UDP / DNS Qry b'teams.events.data.microsoft.com.'
Ether / IP / UDP / DNS Ans b'teams-events-da

<Sniffed: TCP:0 UDP:20 ICMP:0 Other:0>

In [9]:
from scapy.all import sniff
from scapy.layers.inet import TCP, UDP, ICMP

tcp_count = 0
udp_count = 0
icmp_count = 0

packets = sniff(count=50)

for packet in packets:

    if TCP in packet:
        tcp_count += 1

    elif UDP in packet:
        udp_count += 1

    elif ICMP in packet:
        icmp_count += 1

print("TCP :", tcp_count)
print("UDP :", udp_count)
print("ICMP:", icmp_count)

TCP : 7
UDP : 43
ICMP: 0


In [4]:
from scapy.all import sniff, wrpcap

print("Capturing packets...")

packets = sniff(count=50)

wrpcap(
    "capture.pcap",
    packets
)

print("Saved capture.pcap")

Capturing packets...
Saved capture.pcap


In [5]:
from scapy.all import rdpcap

packets = rdpcap(
    "capture.pcap"
)

print(
    "Packets:",
    len(packets)
)

for packet in packets[:10\]:

    print(
        packet.summary()
    )

SyntaxError: unexpected character after line continuation character (3661033236.py, line 12)

In [7]:
from scapy.all import rdpcap

packets = rdpcap("capture.pcap")

print(
    "Packets:",
    len(packets)
)

for packet in packets[:10]:

    print(
        packet.summary()
    )

Packets: 50
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.27:50662 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 PA / Raw


In [8]:
from scapy.all import rdpcap

packets = rdpcap("capture.pcap")

print("=" * 50)
print("PCAP ANALYSIS")
print("=" * 50)

print("Total Packets:", len(packets))
print()

for i, packet in enumerate(packets[:10], start=1):

    print(
        f"Packet {i}: {packet.summary()}"
    )

PCAP ANALYSIS
Total Packets: 50

Packet 1: Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Packet 2: Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Packet 3: Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Packet 4: Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Packet 5: Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Packet 6: Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Packet 7: Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Packet 8: Ether / IP / UDP 51.5.71.27:50662 > 10.0.9.200:61436 / Raw
Packet 9: Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Packet 10: Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 PA / Raw


The error is simply a typo in your code.

### ❌ Your code

```python
for packet in packets[:10\]:
```

The `\` after `10` is invalid Python syntax.

That's why you get:

```text
SyntaxError: unexpected character after line continuation character
```

***

### ✅ Fixed Code

```python
from scapy.all import rdpcap

packets = rdpcap("capture.pcap")

print(
    "Packets:",
    len(packets)
)

for packet in packets[:10]:

    print(
        packet.summary()
    )
```

***

### ⚠️ If You Get Another Error

If you see:

```text
FileNotFoundError:
[Errno 2] No such file or directory: 'capture.pcap'
```

then the PCAP file doesn't exist in your current working directory.

Check:

```python
import os

print(os.getcwd())
```

and make sure `capture.pcap` is located there.

***

### Verify the File Exists

```python
import os

print(os.listdir())
```

You should see something like:

```text
[
 'capture.pcap',
 'test.py',
 ...
]
```

***

### Better Version (Network Engineer Style)

```python
from scapy.all import rdpcap

packets = rdpcap("capture.pcap")

print("=" * 50)
print("PCAP ANALYSIS")
print("=" * 50)

print("Total Packets:", len(packets))
print()

for i, packet in enumerate(packets[:10], start=1):

    print(
        f"Packet {i}: {packet.summary()}"
    )
```

Example Output:

```text
==================================================
PCAP ANALYSIS
==================================================

Total Packets: 150

Packet 1: Ether / IP / TCP 192.168.1.15 > 13.107.246.40
Packet 2: Ether / IP / UDP 192.168.1.15 > 8.8.8.8
Packet 3: Ether / ARP who has 192.168.1.1 says 192.168.1.15
```

### Why this happened

In some of the earlier examples, the text was rendered with:

```python
[:10\]
```

because of markdown escaping. In actual Python code it must always be:

```python
[:10]
```

with **no backslash**.


In [1]:
from scapy.all import sniff

packet = sniff(count=1)[0]

packet.show()

###[ Ethernet ]###
  dst       = 12:34:56:78:9a:bc
  src       = 00:0d:3a:fd:68:b8
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 392
     id        = 33598
     flags     = DF
     frag      = 0
     ttl       = 128
     proto     = udp
     chksum    = 0x0
     src       = 10.0.9.200
     dst       = 51.5.71.36
     \options   \
###[ UDP ]###
        sport     = 61436
        dport     = 3478
        len       = 372
        chksum    = 0x8f76
###[ Raw ]###
           load      = b"@\x00\x01f\x01\xf3y\xcd\x0f8\x04\xc0\x010U\x01\xe0<\x00c\x0c\x17\x03\x03\x01P\xdd SA\xeaB\xe6\xf5\xdc\xb6\xf1\xbd\xe1\x82\x8a\x9a4A\xbfA\x91\xf4\xd8\x10\xd6PQ\x0e\x98\x17\x1a\xa7\xab6>\x8e\x10\x11H!\xa9\xc6^\xda\xc0D\xeb\xa9\xcc2\xd2\xcd#\x82\xdf\x1biY\xffv\xae\xc9\xeaJ%\xf5}\xc9z\x91&\x1a\xf6\xe2\xa1\xd9)I\x9e\x97\x08\x92\x90\xd6\xe6|?\xf6\xa2\xa8\xa4]xM\xe8R]\xe1\xf7\x8ab\xb7\xae\xab,\xe2\xd6S\xa2\x85l\xe0\xf0\x1f\xb9\xafY|\xe8*/p\xa4jt\x10Yj\x8

In [2]:
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP, ICMP

def analyze(packet):

    if IP in packet:

        src = packet[IP].src
        dst = packet[IP].dst

        if TCP in packet:

            print(
                f"TCP  {src} -> {dst}"
            )

        elif UDP in packet:

            print(
                f"UDP  {src} -> {dst}"
            )

        elif ICMP in packet:

            print(
                f"ICMP {src} -> {dst}"
            )

sniff(
    count=100,
    prn=analyze
)

UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  51.5.71.12 -> 10.0.9.200
UDP  10.0.9.200 -> 51.5.71.36
TCP  10.0.9.200 -> 52.112.38.42
UDP  51.5.71.12 -> 10.0.9.200
UDP  51.5.71.12 -> 10.0.9.200
UDP  51.5.71.12 -> 10.0.9.200
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  51.5.71.12 -> 10.0.9.200
UDP  10.0.9.200 -> 51.5.71.36
UDP  51.5.71.12 -> 10.0.9.200
UDP  10.0.9.200 -> 51.5.71.36
TCP  20.14.20.85 -> 10.0.9.200
TCP  10.0.9.200 -> 20.14.20.85
TCP  104.208.170.151 -> 10.0.9.200
UDP  51.5.71.12 -> 10.0.9.200
UDP  142.250.73.106 -> 10.0.9.200
UDP  10.0.9.200 -> 142.250.73.106
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  10.0.9.200 -> 51.5.71.36
UDP  51.5.71.12 -> 10.0.9.200
TCP  104.208.170.151 -> 10.0.9.200
TCP  104.208.170.151 -> 10.0.9.200
TCP  104.208.170.151 -> 10.0.9.200
TCP  104.208.170.151 -> 10.0.9.200
TCP

<Sniffed: TCP:46 UDP:54 ICMP:0 Other:0>

In [ ]:
from scapy.all import sniff

packets = sniff(
    timeout=30
)

print(
    "Total packets:",
    len(packets)
)

In [ ]:
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP, ICMP

tcp = 0
udp = 0
icmp = 0

def analyze(packet):
    global tcp
    global udp
    global icmp

    if TCP in packet:
        tcp += 1

    elif UDP in packet:
        udp += 1

    elif ICMP in packet:
        icmp += 1

sniff(
    count=100,
    prn=analyze
)

print("\nTraffic Summary")
print("-" * 30)
print("TCP :", tcp)
print("UDP :", udp)
print("ICMP:", icmp)

Perfect. Since you're using **VS Code** and learning networking, I recommend **Scapy** instead of PyShark for now. It avoids the Jupyter event-loop problems and teaches packet analysis directly. Scapy is a Python-based packet manipulation and capture library that supports sniffing, reading/writing pcap files, packet parsing, filtering, and protocol analysis. [\[github.com\]](https://github.com/secdev/scapy), [\[0xbharath.github.io\]](https://0xbharath.github.io/art-of-packet-crafting-with-scapy/scapy/sniffing/index.html)

# Step 1: Install Scapy

```bash
pip install scapy
```

Verify:

```bash
pip show scapy
```

***

# Example 1: Your First Packet Capture

Create:

```python
from scapy.all import sniff

def show_packet(packet):
    print(packet.summary())

sniff(
    count=10,
    prn=show_packet
)
```

Output:

```text
Ether / IP / TCP 192.168.1.10:52000 > 13.107.246.40:https
Ether / IP / UDP 192.168.1.10 > 8.8.8.8:dns
```

Learn:

* Functions
* Packet objects
* Callback functions

***

# Example 2: Capture Only ICMP (Ping)

Run the script:

```python
from scapy.all import sniff

sniff(
    filter="icmp",
    count=10,
    prn=lambda pkt: print(pkt.summary())
)
```

Open another terminal:

```bash
ping 8.8.8.8
```

Output:

```text
IP / ICMP 192.168.1.10 > 8.8.8.8 echo-request
IP / ICMP 8.8.8.8 > 192.168.1.10 echo-reply
```

This is a great way to learn packet flows.

***

# Example 3: Print Source and Destination IPs

```python
from scapy.all import sniff, IP

def analyze(packet):

    if IP in packet:

        print(
            "Source:",
            packet[IP].src,
            "->",
            "Destination:",
            packet[IP].dst
        )

sniff(
    count=20,
    prn=analyze
)
```

Output:

```text
Source: 192.168.1.15 -> Destination: 8.8.8.8
Source: 8.8.8.8 -> Destination: 192.168.1.15
```

***

# Example 4: Display TCP Ports

```python
from scapy.all import sniff, IP, TCP

def analyze(packet):

    if TCP in packet:

        print(
            packet[IP].src,
            ":",
            packet[TCP].sport,
            "-->",
            packet[IP].dst,
            ":",
            packet[TCP].dport
        )

sniff(
    filter="tcp",
    count=20,
    prn=analyze
)
```

Output:

```text
192.168.1.15 : 55231 --> 13.107.246.40 : 443
```

Learn:

* Source Port
* Destination Port
* HTTPS traffic

***

# Example 5: DNS Packet Sniffer

```python
from scapy.all import sniff

sniff(
    filter="udp port 53",
    count=20,
    prn=lambda pkt: print(pkt.summary())
)
```

Open browser:

```text
www.microsoft.com
```

Output:

```text
IP / UDP / DNS Query
IP / UDP / DNS Response
```

***

# Example 6: Count TCP, UDP and ICMP Packets

```python
from scapy.all import sniff
from scapy.layers.inet import TCP, UDP, ICMP

tcp_count = 0
udp_count = 0
icmp_count = 0

packets = sniff(count=50)

for packet in packets:

    if TCP in packet:
        tcp_count += 1

    elif UDP in packet:
        udp_count += 1

    elif ICMP in packet:
        icmp_count += 1

print("TCP :", tcp_count)
print("UDP :", udp_count)
print("ICMP:", icmp_count)
```

Example:

```text
TCP : 34
UDP : 12
ICMP: 4
```

***

# Example 7: Save Packets to a PCAP

```python
from scapy.all import sniff, wrpcap

print("Capturing packets...")

packets = sniff(count=50)

wrpcap(
    "capture.pcap",
    packets
)

print("Saved capture.pcap")
```

Now open:

```text
capture.pcap
```

in Wireshark.

Scapy supports reading and writing packet captures in pcap format. [\[geeksforgeeks.org\]](https://www.geeksforgeeks.org/python/packet-sniffing-using-scapy/), [\[github.com\]](https://github.com/secdev/scapy)

***

# Example 8: Read a PCAP File

```python
from scapy.all import rdpcap

packets = rdpcap(
    "capture.pcap"
)

print(
    "Packets:",
    len(packets)
)

for packet in packets[:10]:

    print(
        packet.summary()
    )
```

Output:

```text
Packets: 50

Ether / IP / TCP
Ether / IP / UDP
```

***

# Example 9: Show Packet Details

```python
from scapy.all import sniff

packet = sniff(count=1)[0]

packet.show()
```

Output:

```text
###[ Ethernet ]###
dst=00:11:22:33:44:55
src=66:77:88:99:AA:BB

###[ IP ]###
src=192.168.1.10
dst=8.8.8.8

###[ TCP ]###
sport=55220
dport=443
```

This is similar to Wireshark packet details.

***

# Example 10: Mini Network Analyzer

```python
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP, ICMP

def analyze(packet):

    if IP in packet:

        src = packet[IP].src
        dst = packet[IP].dst

        if TCP in packet:

            print(
                f"TCP  {src} -> {dst}"
            )

        elif UDP in packet:

            print(
                f"UDP  {src} -> {dst}"
            )

        elif ICMP in packet:

            print(
                f"ICMP {src} -> {dst}"
            )

sniff(
    count=100,
    prn=analyze
)
```

Example:

```text
TCP 192.168.1.15 -> 13.107.246.40
UDP 192.168.1.15 -> 8.8.8.8
ICMP 192.168.1.15 -> 8.8.8.8
```

***

# Example 11: Display HTTP/HTTPS Connections

```python
from scapy.all import sniff
from scapy.layers.inet import IP, TCP

def analyze(packet):

    if TCP in packet:

        if packet[TCP].dport in [80, 443]:

            print(
                f"{packet[IP].src} -> "
                f"{packet[IP].dst} "
                f"Port {packet[TCP].dport}"
            )

sniff(
    filter="tcp",
    count=50,
    prn=analyze
)
```

Output:

```text
192.168.1.15 -> 13.107.246.40 Port 443
```

***

# Example 12: Capture for 30 Seconds

```python
from scapy.all import sniff

packets = sniff(
    timeout=30
)

print(
    "Total packets:",
    len(packets)
)
```

Scapy supports limiting captures by packet count or timeout. [\[0xbharath.github.io\]](https://0xbharath.github.io/art-of-packet-crafting-with-scapy/scapy/sniffing/index.html), [\[geeksforgeeks.org\]](https://www.geeksforgeeks.org/python/packet-sniffing-using-scapy/)

***

# Real Network Engineer Practice Project

This is the project I'd recommend for you next:

```python
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP, ICMP

tcp = 0
udp = 0
icmp = 0

def analyze(packet):
    global tcp
    global udp
    global icmp

    if TCP in packet:
        tcp += 1

    elif UDP in packet:
        udp += 1

    elif ICMP in packet:
        icmp += 1

sniff(
    count=100,
    prn=analyze
)

print("\nTraffic Summary")
print("-" * 30)
print("TCP :", tcp)
print("UDP :", udp)
print("ICMP:", icmp)
```

This combines:

* Variables
* Functions
* Loops
* Protocol identification
* Packet analysis
* Real-world troubleshooting concepts

It's very similar to the first step of building a basic packet analyzer used in network automation and troubleshooting.


In [1]:
from scapy.all import sniff, wrpcap

print("Capturing 50 packets...")

packets = sniff(count=50)

wrpcap(
    "capture123354.pcap",
    packets
)

print("Saved capture.pcap")

Capturing 50 packets...
Saved capture.pcap


In [2]:
from scapy.all import sniff, wrpcap

print("Capturing for 30 seconds...")

packets = sniff(timeout=30)

wrpcap(
    "30sec_capture.pcap",
    packets
)

print("Capture saved.")

Capturing for 30 seconds...
Capture saved.


In [3]:
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="icmp",
    timeout=20
)

wrpcap(
    "icmp_capture.pcap",
    packets
)

print("ICMP packets saved")

ICMP packets saved


In [4]:
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="udp port 53",
    timeout=20
)

wrpcap(
    "dns_capture.pcap",
    packets
)

print("DNS capture completed")

DNS capture completed


In [5]:
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="tcp port 443",
    timeout=30
)

wrpcap(
    "https_capture.pcap",
    packets
)

print("HTTPS packets saved")

HTTPS packets saved


In [6]:
from scapy.all import sniff, wrpcap
from datetime import datetime

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

filename = f"capture_{timestamp}.pcap"

packets = sniff(
    count=100
)

wrpcap(
    filename,
    packets
)

print(
    f"Saved {filename}"
)

Saved capture_20260712_121648.pcap


In [7]:
from scapy.all import sniff, wrpcap

captured = []

def collect(packet):
    captured.append(packet)

    print(
        f"Packet #{len(captured)}"
    )

sniff(
    count=20,
    prn=collect
)

wrpcap(
    "counter_capture.pcap",
    captured
)

print("Saved pcap file")

Packet #1
Packet #2
Packet #3
Packet #4
Packet #5
Packet #6
Packet #7
Packet #8
Packet #9
Packet #10
Packet #11
Packet #12
Packet #13
Packet #14
Packet #15
Packet #16
Packet #17
Packet #18
Packet #19
Packet #20
Saved pcap file


In [9]:
from scapy.all import sniff, wrpcap

packets = sniff(
    count=10
)

for packet in packets:

    print(
        packet.summary()
    )

wrpcap(
    "summary_capture.pcap",
    packets
)

print("Capture saved")


Ether / IP / TCP 52.107.248.19:https > 10.0.9.200:62567 A / Padding
Ether / IP / TCP 10.0.9.200:62567 > 52.107.248.19:https A
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.12:50818 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.12:50818 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.12:50818 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / TCP 52.123.128.14:https > 10.0.9.200:60033 A / Padding
Ether / IP / TCP 10.0.9.200:60033 > 52.123.128.14:https A
Capture saved


In [10]:
from scapy.all import rdpcap

packets = rdpcap(
    "capture.pcap"
)

print(
    "Total Packets:",
    len(packets)
)

for packet in packets[:10]:

    print(
        packet.summary()
    )

Total Packets: 50
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 10.0.9.200:61436 > 51.5.71.36:3478 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.12:50818 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.27:50662 > 10.0.9.200:61436 / Raw
Ether / IP / UDP 51.5.71.36:3478 > 10.0.9.200:61436 / Raw
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 PA / Raw


In [11]:
import os

print(os.listdir())

['30sec_capture.pcap', 'capture.pcap', 'capture123354.pcap', 'capture_20260712_121648.pcap', 'counter_capture.pcap', 'device_information.ipynb', 'dns_capture.pcap', 'https_capture.pcap', 'icmp_capture.pcap', 'speed_test.ipynb', 'summary_capture.pcap', 'system_report.txt', 'wifi_status.ipynb', 'wire Shark & packet captures.ipynb']


Absolutely. Saving packets to a **PCAP file** is one of the most useful Scapy skills because you can later open the file in **Wireshark** and analyze the traffic.

***

# Example 1: Capture 50 Packets and Save to PCAP

```python
from scapy.all import sniff, wrpcap

print("Capturing 50 packets...")

packets = sniff(count=50)

wrpcap(
    "capture.pcap",
    packets
)

print("Saved capture.pcap")
```

Output:

```text
Capturing 50 packets...
Saved capture.pcap
```

Creates:

```text
capture.pcap
```

***

# Example 2: Capture Packets for 30 Seconds

```python
from scapy.all import sniff, wrpcap

print("Capturing for 30 seconds...")

packets = sniff(timeout=30)

wrpcap(
    "30sec_capture.pcap",
    packets
)

print("Capture saved.")
```

Output:

```text
Capture saved.
```

***

# Example 3: Capture Only ICMP (Ping) Traffic

Start the script:

```python
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="icmp",
    timeout=20
)

wrpcap(
    "icmp_capture.pcap",
    packets
)

print("ICMP packets saved")
```

Then open another terminal:

```bash
ping 8.8.8.8
```

***

# Example 4: Capture DNS Packets

```python
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="udp port 53",
    timeout=20
)

wrpcap(
    "dns_capture.pcap",
    packets
)

print("DNS capture completed")
```

Visit a website while the script is running and DNS queries will be captured.

***

# Example 5: Capture HTTPS Traffic

```python
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="tcp port 443",
    timeout=30
)

wrpcap(
    "https_capture.pcap",
    packets
)

print("HTTPS packets saved")
```

***

# Example 6: Timestamp the PCAP Filename

Useful for automation.

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

filename = f"capture_{timestamp}.pcap"

packets = sniff(
    count=100
)

wrpcap(
    filename,
    packets
)

print(
    f"Saved {filename}"
)
```

Example:

```text
capture_20260712_143015.pcap
```

***

# Example 7: Live Capture with Packet Counter

```python
from scapy.all import sniff, wrpcap

captured = []

def collect(packet):
    captured.append(packet)

    print(
        f"Packet #{len(captured)}"
    )

sniff(
    count=20,
    prn=collect
)

wrpcap(
    "counter_capture.pcap",
    captured
)

print("Saved pcap file")
```

***

# Example 8: Capture and Display Summary Before Saving

```python
from scapy.all import sniff, wrpcap

packets = sniff(
    count=10
)

for packet in packets:

    print(
        packet.summary()
    )

wrpcap(
    "summary_capture.pcap",
    packets
)

print("Capture saved")
```

***

# Example 9: Read Back the Saved PCAP

After saving:

```python
from scapy.all import rdpcap

packets = rdpcap(
    "capture.pcap"
)

print(
    "Total Packets:",
    len(packets)
)

for packet in packets[:10]:

    print(
        packet.summary()
    )
```

Expected:

```text
Total Packets: 50

Ether / IP / TCP ...
Ether / IP / UDP ...
```

***

# Example 10: Network Engineer Packet Capture Tool

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

print("=" * 50)
print("NETWORK PACKET CAPTURE TOOL")
print("=" * 50)

packet_count = 100

packets = sniff(
    count=packet_count
)

filename = (
    "network_capture_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

wrpcap(
    filename,
    packets
)

print()
print("Packets Captured:", len(packets))
print("File Saved:", filename)
```

Example Output:

```text
==================================================
NETWORK PACKET CAPTURE TOOL
==================================================

Packets Captured: 100
File Saved: network_capture_20260712_143500.pcap
```

***

# Verify the File Exists

After saving:

```python
import os

print(os.listdir())
```

Example:

```text
[
 'capture.pcap',
 'dns_capture.pcap',
 'network_capture_20260712_143500.pcap'
]
```

***

# Open the PCAP in Wireshark

From terminal:

```bash
wireshark capture.pcap
```

or simply double-click:

```text
capture.pcap
```

You can then inspect:

* Source IP
* Destination IP
* TCP flags
* DNS queries
* ICMP packets
* HTTP/HTTPS metadata
* Packet timing

For learning network automation, I'd recommend starting with a **DNS capture project**, then moving to **ICMP**, then building a script that captures traffic and generates a summary report (TCP/UDP/ICMP counts) before saving the PCAP.


In [12]:
packets = sniff(count=100)

wrpcap("capture.pcap", packets)

In [13]:
from scapy.all import sniff, wrpcap
from datetime import datetime

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

filename = f"capture_{timestamp}.pcap"

packets = sniff(count=50)

wrpcap(filename, packets)

print("Saved:", filename)

Saved: capture_20260712_121852.pcap


In [14]:
from scapy.all import sniff, wrpcap
from datetime import datetime
import os

folder = datetime.now().strftime("%Y-%m-%d")

os.makedirs(folder, exist_ok=True)

filename = os.path.join(
    folder,
    datetime.now().strftime("%H%M%S.pcap")
)

packets = sniff(count=100)

wrpcap(filename, packets)

print("Saved:", filename)

Saved: 2026-07-12\121910.pcap


In [ ]:
from scapy.all import sniff, wrpcap
from datetime import datetime
import time

while True:

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    filename = f"capture_{timestamp}.pcap"

    packets = sniff(timeout=300)

    wrpcap(filename, packets)

    print("Saved:", filename)

Saved: capture_20260712_121928.pcap


In [1]:
packets = sniff(
    filter="icmp",
    timeout=20
)

NameError: name 'sniff' is not defined

In [3]:
from scapy import all
packets = sniff(
    filter="tcp port 443",
    timeout=30
)

NameError: name 'sniff' is not defined

In [6]:
from scapy.all import sniff

packets = sniff(
    filter="icmp",
    timeout=20
)

print("Packets Captured:", len(packets))

Packets Captured: 4


In [4]:
from scapy.all import sniff, wrpcap
from datetime import datetime

filename = (
    "dns_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

packets = sniff(
    filter="udp port 53",
    timeout=30
)

wrpcap(filename, packets)

print("Saved:", filename)

Saved: dns_20260712_122117.pcap


In [5]:
from scapy.all import sniff, wrpcap
from datetime import datetime

print("=" * 50)
print("DNS TROUBLESHOOTING CAPTURE")
print("=" * 50)

filename = (
    "dns_capture_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

packets = sniff(
    filter="udp port 53",
    timeout=60
)

wrpcap(filename, packets)

print()
print("Packets Captured:", len(packets))
print("Saved File:", filename)

DNS TROUBLESHOOTING CAPTURE

Packets Captured: 62
Saved File: dns_capture_20260712_122151.pcap


In [7]:
from scapy.all import sniff, wrpcap
from datetime import datetime

print("=" * 50)
print("DNS TROUBLESHOOTING CAPTURE")
print("=" * 50)

filename = (
    "dns_capture_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

packets = sniff(
    filter="udp port 53",
    timeout=60
)

wrpcap(filename, packets)

print()
print("Packets Captured:", len(packets))
print("Saved File:", filename)

DNS TROUBLESHOOTING CAPTURE

Packets Captured: 84
Saved File: dns_capture_20260712_122311.pcap


Great question. This is exactly how packet captures are automated in network operations: capture traffic automatically, save the file with a timestamp, and optionally filter only the traffic you're interested in.

***

# How Packet Capture Automation Works

Instead of:

```python
packets = sniff(count=100)

wrpcap("capture.pcap", packets)
```

which overwrites the same file every time,

you generate a unique filename using the current date and time.

### Example

```python
capture_20260712_104501.pcap
capture_20260712_104630.pcap
capture_20260712_104755.pcap
```

This allows:

* Historical analysis
* Troubleshooting later
* Comparing captures
* Automated evidence collection

***

# Example 1: Save Capture with Timestamp

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

filename = f"capture_{timestamp}.pcap"

packets = sniff(count=50)

wrpcap(filename, packets)

print("Saved:", filename)
```

### Example Output

```text
Saved: capture_20260712_104501.pcap
```

***

# Example 2: Automatically Create Daily Folder

```python
from scapy.all import sniff, wrpcap
from datetime import datetime
import os

folder = datetime.now().strftime("%Y-%m-%d")

os.makedirs(folder, exist_ok=True)

filename = os.path.join(
    folder,
    datetime.now().strftime("%H%M%S.pcap")
)

packets = sniff(count=100)

wrpcap(filename, packets)

print("Saved:", filename)
```

### Result

```text
2026-07-12/
    104530.pcap
    104810.pcap
```

***

# Example 3: Continuous Packet Capture

Capture every 5 minutes automatically.

```python
from scapy.all import sniff, wrpcap
from datetime import datetime
import time

while True:

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    filename = f"capture_{timestamp}.pcap"

    packets = sniff(timeout=300)

    wrpcap(filename, packets)

    print("Saved:", filename)
```

### Result

```text
capture_20260712_103000.pcap
capture_20260712_103500.pcap
capture_20260712_104000.pcap
```

Very useful for long troubleshooting sessions.

***

# Packet Filtering During Capture

Without a filter:

```python
sniff()
```

captures:

```text
ARP
TCP
UDP
ICMP
DNS
HTTPS
Everything
```

Filtering lets you capture only what interests you.

***

# Example 4: ICMP Filter

Capture only ping traffic.

```python
packets = sniff(
    filter="icmp",
    timeout=20
)
```

Generate traffic:

```bash
ping 8.8.8.8
```

Captured:

```text
Echo Request
Echo Reply
```

***

# Example 5: DNS Filter

Capture only DNS traffic.

```python
packets = sniff(
    filter="udp port 53",
    timeout=30
)
```

Visit:

```text
www.microsoft.com
```

Captured:

```text
DNS Query
DNS Response
```

***

# Example 6: HTTPS Traffic

Capture secure web traffic metadata.

```python
packets = sniff(
    filter="tcp port 443",
    timeout=30
)
```

Captured:

```text
TLS
HTTPS
Encrypted web traffic
```

***

# Example 7: HTTP Traffic

```python
packets = sniff(
    filter="tcp port 80",
    timeout=30
)
```

Captured:

```text
HTTP requests
HTTP responses
```

***

# Example 8: Specific Host

Capture only traffic involving Google DNS.

```python
packets = sniff(
    filter="host 8.8.8.8",
    timeout=30
)
```

Captured:

```text
192.168.1.10 -> 8.8.8.8
8.8.8.8 -> 192.168.1.10
```

***

# Example 9: Source IP Filter

```python
packets = sniff(
    filter="src host 192.168.1.15",
    timeout=30
)
```

Only packets originating from:

```text
192.168.1.15
```

***

# Example 10: Destination IP Filter

```python
packets = sniff(
    filter="dst host 8.8.8.8",
    timeout=30
)
```

Only packets going to:

```text
8.8.8.8
```

***

# Example 11: TCP Only

```python
packets = sniff(
    filter="tcp",
    timeout=30
)
```

Captured:

```text
SSH
HTTPS
HTTP
BGP
Many TCP protocols
```

***

# Example 12: UDP Only

```python
packets = sniff(
    filter="udp",
    timeout=30
)
```

Captured:

```text
DNS
DHCP
NTP
SNMP
```

***

# Example 13: Combine Filters

Capture DNS from a specific server:

```python
packets = sniff(
    filter="udp port 53 and host 8.8.8.8",
    timeout=30
)
```

***

# Example 14: Port Range

Capture web traffic.

```python
packets = sniff(
    filter="tcp portrange 80-443",
    timeout=30
)
```

***

# Example 15: Save Filtered Capture

Capture only DNS and write directly to a timestamped PCAP.

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

filename = (
    "dns_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

packets = sniff(
    filter="udp port 53",
    timeout=30
)

wrpcap(filename, packets)

print("Saved:", filename)
```

Example:

```text
Saved: dns_20260712_105512.pcap
```

***

# Real Network Engineer Example

This is very similar to what a WAN engineer might do when troubleshooting DNS, reachability, or application connectivity.

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

print("=" * 50)
print("DNS TROUBLESHOOTING CAPTURE")
print("=" * 50)

filename = (
    "dns_capture_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".pcap"
)

packets = sniff(
    filter="udp port 53",
    timeout=60
)

wrpcap(filename, packets)

print()
print("Packets Captured:", len(packets))
print("Saved File:", filename)
```

### Skills learned

* Packet capture
* BPF filtering
* File handling
* Time stamping
* Automation
* Network troubleshooting workflow
* Generating Wireshark-readable PCAPs

As your next step, I'd recommend building a **packet capture + protocol summary tool** that automatically counts TCP, UDP, ICMP, DNS, and ARP packets, then saves both a `.pcap` file and a text report.
